# GNN-BERT Music Context: Real-Data Demo

Runs a short, real training pass for each of the 4 tasks on your **actual cached GTZAN and MagnaTagATune data** (not synthetic), and shows one real inference example per task.

**Prerequisites** — run these once from `src/` before this notebook, if you haven't already:
```
python prepare_gtzan.py --config ../config.yaml
python prepare_magnatagatune.py --config ../config.yaml --annotations <path> --audio_root <path> --max_per_split 600 100 100
```

Each task below trains briefly (a handful of epochs) since no model checkpoint is saved anywhere in this project — every training script (including this notebook) starts from a freshly initialized model each time. For the full training runs and real reported metrics, see `results/*.json`, produced by the longer runs described in `README.md`.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
import torch

SPLITS_DIR = os.path.abspath('../data/splits')
gtzan_ready = os.path.exists(os.path.join(SPLITS_DIR, 'gtzan_split.json'))
mtt_ready = os.path.exists(os.path.join(SPLITS_DIR, 'mtt_split.json'))
print('GTZAN prepared:', gtzan_ready)
print('MagnaTagATune prepared:', mtt_ready)
if not (gtzan_ready and mtt_ready):
    print('\nRun the prerequisite prepare_*.py scripts above before continuing.')

## Task 2: GNN Genre Classifier (real GTZAN)

In [ ]:
import train_gtzan as TG
from gnn_model import GNNTagClassifier
import torch.nn.functional as F
import numpy as np

split = TG.load_split(SPLITS_DIR)
train_graphs, train_labels = TG.load_graphs(split['train'])
test_graphs, test_labels = TG.load_graphs(split['test'])

in_dim = train_graphs[0].x.shape[1]
gtzan_model = GNNTagClassifier(in_dim=in_dim, num_tags=len(TG.GENRES), hidden_dim=64, num_layers=3)
opt = torch.optim.Adam(gtzan_model.parameters(), lr=1e-3)

print('Training GNN briefly (5 epochs) on real GTZAN graphs...')
for epoch in range(5):
    gtzan_model.train()
    for g, y in TG.batches(train_graphs, train_labels, 16):
        logits = gtzan_model(g.x, g.edge_index, g.batch)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()

acc, f1 = TG.evaluate(gtzan_model, test_graphs, test_labels)
print(f'Test accuracy={acc:.3f}  macro_f1={f1:.3f}  (short demo run; see results/gtzan_real_results.json for the full 15-epoch reported numbers)')

In [ ]:
# One real inference example
from torch_geometric.data import Batch
idx = 0
rec = split['test'][idx]
with torch.no_grad():
    g = Batch.from_data_list([test_graphs[idx]])
    probs = F.softmax(gtzan_model(g.x, g.edge_index, g.batch), dim=-1).squeeze(0)
pred = TG.GENRES[probs.argmax().item()]
print(f"Track: {rec['track_id']}")
print(f"True genre: {rec['genre']}")
print(f"Predicted genre: {pred}  (confidence {probs.max().item():.3f})")

## Task 1: BERT Tag Classifier (real MagnaTagATune)

In [ ]:
import train_mtt_task1 as T1
from bert_encoder import BertTagClassifier

mtt_data = T1.load_split(SPLITS_DIR)
tag_vocab = mtt_data['tag_vocab']
mtt_split = mtt_data['splits']

print('Loading bert-base-uncased (downloads on first run, needs internet)...')
bert_model = BertTagClassifier(num_tags=len(tag_vocab), synthetic=False, freeze_encoder=True)
opt = torch.optim.Adam(bert_model.parameters(), lr=1e-3)

print('Training briefly (3 epochs, frozen encoder)...')
for epoch in range(3):
    bert_model.train()
    for batch in T1.batches(mtt_split['train'], 16, shuffle=True):
        ids, mask = T1.encode_batch(bert_model, batch, max_len=128)
        y = T1.label_tensor(batch)
        loss = F.binary_cross_entropy_with_logits(bert_model(ids, mask), y)
        opt.zero_grad(); loss.backward(); opt.step()

macro, micro, aucpr = T1.evaluate(bert_model, mtt_split['test'], batch_size=16, max_len=128)
print(f'Test macro_f1={macro:.3f} micro_f1={micro:.3f} aucpr={aucpr:.3f} (short demo run; see results/mtt_real_results.json for the full 5-epoch reported numbers)')

In [ ]:
# One real inference example
rec = mtt_split['test'][0]
with torch.no_grad():
    ids, mask = T1.encode_batch(bert_model, [rec], max_len=128)
    probs = torch.sigmoid(bert_model(ids, mask)).squeeze(0)
top5 = torch.topk(probs, k=5).indices.tolist()
print(f"Clip: {rec['clip_id']}")
print(f"True tags: {rec['tags']}")
print(f"Predicted top-5 tags: {[tag_vocab[i] for i in top5]}")

## Task 3: GNN-BERT Cross-Attention Fusion (real MagnaTagATune)

In [ ]:
import train_mtt_fusion as T3

fusion_model = T3.build_model('cross_attention', len(tag_vocab), {
    'gnn': {'hidden_dim': 64, 'num_layers': 3, 'encoder': 'graphsage', 'dropout': 0.2},
    'bert': {'pretrained_name': 'bert-base-uncased'},
    'fusion': {'attn_heads': 4},
}, freeze_encoder=True)
opt = torch.optim.Adam((p for p in fusion_model.parameters() if p.requires_grad), lr=1e-3)

print('Training briefly (3 epochs, frozen encoder)...')
for epoch in range(3):
    fusion_model.train()
    for batch in T3.batches(mtt_split['train'], 16, shuffle=True):
        logits = T3.forward_pass(fusion_model, 'cross_attention', batch, 128, fusion_model.bert)
        y = T3.label_tensor(batch)
        loss = F.binary_cross_entropy_with_logits(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()

macro, micro, aucpr = T3.evaluate(fusion_model, 'cross_attention', mtt_split['test'], 16, 128, fusion_model.bert)
print(f'Test macro_f1={macro:.3f} micro_f1={micro:.3f} aucpr={aucpr:.3f} (short demo run; see results/mtt_real_results.json for the full 8-epoch 4-way ablation)')

In [ ]:
# One real inference example
rec = mtt_split['test'][0]
with torch.no_grad():
    logits = T3.forward_pass(fusion_model, 'cross_attention', [rec], 128, fusion_model.bert)
    probs = torch.sigmoid(logits).squeeze(0)
top5 = torch.topk(probs, k=5).indices.tolist()
print(f"Clip: {rec['clip_id']}")
print(f"True tags: {rec['tags']}")
print(f"Fusion model predicted top-5 tags: {[tag_vocab[i] for i in top5]}")

## Task 4: Contrastive Retrieval (real MagnaTagATune)

In [ ]:
import train_mtt_contrastive as T4
from contrastive import DualEncoder, info_nce_loss, retrieval_recall_at_k, top_k_matches

dual_model = DualEncoder(graph_in_dim=32, embedding_dim=128, bert_synthetic=False,
                          gnn_hidden=64, gnn_layers=3)
for p in dual_model.bert.bert.parameters():
    p.requires_grad = False
opt = torch.optim.Adam((p for p in dual_model.parameters() if p.requires_grad), lr=1e-3)

print('Training briefly (5 epochs, frozen encoder)...')
for epoch in range(5):
    dual_model.train()
    for batch in T4.batches(mtt_split['train'], 32, shuffle=True):
        from torch_geometric.data import Batch as PygBatch
        g_data = PygBatch.from_data_list([T4.load_graph(r) for r in batch])
        ids, mask = dual_model.bert.tokenize([T4.caption_text(r) for r in batch], max_len=128)
        g_emb, t_emb = dual_model(g_data.x, g_data.edge_index, g_data.batch, ids, mask)
        loss = info_nce_loss(g_emb, t_emb, 0.07)
        opt.zero_grad(); loss.backward(); opt.step()

g_test, t_test = T4.encode_split(dual_model, mtt_split['test'], 32, 128)
metrics = retrieval_recall_at_k(g_test, t_test)
print('Test retrieval:', {k: round(v, 3) for k, v in metrics.items()})
print('(short demo run; see results/mtt_real_results.json for the full 10-epoch reported numbers)')

In [ ]:
# One real retrieval example
sim = g_test @ t_test.t()
query_idx = 0
top3 = top_k_matches(sim[query_idx], k=3)
query_rec = mtt_split['test'][query_idx]
print(f"Query caption (from clip {query_rec['clip_id']}): '{T4.caption_text(query_rec)}'")
print(f"Top-3 retrieved audio clip IDs: {[mtt_split['test'][j]['clip_id'] for j in top3]}")
print(f"Correct clip in top-3: {query_rec['clip_id'] in [mtt_split['test'][j]['clip_id'] for j in top3]}")